# 03 — Graph Construction & Network EDA

## Objective
Construct the heterogeneous transaction→user/device/IP graph and quantify connectivity, shared infrastructure, and concentration.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Build the heterogeneous graph

In [1]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
import networkx as nx
from graph_utils import build_bipartite_entity_graph
fraud,_=load_data()
# NetworkX is used for EDA on a deterministic sample to keep memory bounded.
graph_df=fraud.sort_values("purchase_time").iloc[::max(1,len(fraud)//60000)].copy()
G=build_bipartite_entity_graph(graph_df)
display(Markdown(f"**Graph EDA sample:** {len(graph_df):,} transactions | **Nodes:** {G.number_of_nodes():,} | **Edges:** {G.number_of_edges():,} | **Components:** {nx.number_connected_components(G):,}"))


**Graph EDA sample:** 75,556 transactions | **Nodes:** 293,959 | **Edges:** 226,668 | **Components:** 70,706

## 2. Network structure

In [2]:
nt=pd.Series(nx.get_node_attributes(G,"node_type")).value_counts().rename_axis("node_type").to_frame("nodes"); display(nt); deg=pd.DataFrame([(n,G.degree(n),G.nodes[n].get("node_type")) for n in G if G.nodes[n].get("node_type")!="transaction"],columns=["entity","degree","type"]).sort_values("degree",ascending=False); display(deg.head(25)); px.bar(deg.head(15),x="degree",y="entity",color="type",orientation="h",title="Highest-connectivity entities").show()

,nodes
node_type,
transaction,75556
user,75556
ip,72141
device,70706


,entity,degree,type
2297,ip::1502818419.73176,11,ip
2296,device::ZUSVMDEZRBDTX,11,device
3083,device::BWSMVSLCJXMCM,10,device
5430,device::QVMVTZOIJDKNR,10,device
1119,device::NGQCKIADMZORL,10,device
1120,ip::2050963888.16442,10,ip
3397,ip::2141691947.61474,10,ip
5431,ip::235431826.927454,10,ip
2354,ip::2586669382.02567,10,ip
4327,device::IGKYVZDBEGALB,10,device


## 3. Interactive entity lens

In [3]:
tw=widgets.Dropdown(options=["user","device","ip"],value="device",description="Entity"); md=widgets.IntSlider(value=5,min=1,max=100,description="Min degree"); out=widgets.Output()
def show(*_):
    with out: out.clear_output(); display(deg[(deg.type==tw.value)&(deg.degree>=md.value)].head(50))
tw.observe(show,'value'); md.observe(show,'value'); display(widgets.HBox([tw,md]),out); show(); save_json({"nodes":G.number_of_nodes(),"edges":G.number_of_edges()},REP/"graph_profile.json")

Output()